In [5]:
from dotenv import load_dotenv
import os
from sshtunnel import SSHTunnelForwarder
from pymongo import MongoClient


# .envファイルをロードして環境変数を読み込む
load_dotenv()

# SSH接続情報（.envから読み込み）
SSH_HOST = os.getenv('SSH_HOST')
SSH_PORT = int(os.getenv('SSH_PORT', 22))
SSH_USERNAME = os.getenv('SSH_USERNAME')
SSH_PASSWORD = os.getenv('SSH_PASSWORD')

# MongoDB接続情報（.envから読み込み）
MONGO_HOST = os.getenv('MONGO_HOST', '127.0.0.1')
MONGO_PORT = int(os.getenv('MONGO_PORT', 27017))
DATABASE_NAME = 'test'
COLLECTION_NAME = 'collection'

def ssh_conection():
    # SSHトンネルの確立
    with SSHTunnelForwarder(
        (SSH_HOST, SSH_PORT),
        ssh_username=SSH_USERNAME,
        ssh_password=SSH_PASSWORD,
        remote_bind_address=(MONGO_HOST, MONGO_PORT)
    ) as tunnel:
        local_port = tunnel.local_bind_port
        print(f"SSHトンネル確立: ローカルポート {local_port} がリモートの {MONGO_HOST}:{MONGO_PORT} にマッピングされました。")
        
        # MongoDBへの接続（SSHトンネル経由）
        client = MongoClient('127.0.0.1', local_port)
        db = client[DATABASE_NAME]
        collection = db[COLLECTION_NAME]

def write_dataframe_to_mongo(collection, df):
    """
    Pandas DataFrame を受け取り、指定された MongoDB コレクションに一括で書き込むメソッド
    
    Parameters:
        collection (pymongo.collection.Collection): MongoDB のコレクションオブジェクト
        df (pandas.DataFrame): 挿入対象のDataFrame
        
    Returns:
        list: 挿入されたドキュメントのIDリスト
    """
    # DataFrame を辞書のリストに変換
    records = df.to_dict(orient='records')
    
    # insert_manyを使用して一括挿入
    result = collection.insert_many(records)
    return result.inserted_ids

if __name__ == '__main__':
    ssh_conection()
    write_dataframe_to_mongo(COLLECTION_NAME,)

SSHトンネル確立: ローカルポート 53887 がリモートの 127.0.0.1:27017 にマッピングされました。
挿入されたドキュメントのID: 67b34c47e18954c33161a0d3
参照結果: {'_id': ObjectId('67b34b12e18954c33161a0d1'), 'name': 'Alice', 'age': 30, 'city': 'Tokyo'}
